# Route Resilience — GPU training (Colab / Kaggle)

Trains three models, then compares them so the topology advantage is visible:

1. **U-Net** (Dice+Focal) — the simple pixel-loss control.
2. **D-LinkNet** (Dice+Focal) — the DeepGlobe-winning strong CNN baseline; *"what competitors will use"*.
3. **SegFormer-B2 + clDice** — our topology-aware USP model, evaluated with **multi-scale + flip TTA**.

The money table shows IoU/Dice roughly comparable while **clDice, connectivity-ratio and APLS** favour the SegFormer+clDice model — that is the USP, measured against the *strongest* baseline (D-LinkNet), not just U-Net.

**Before running:** Runtime → Change runtime type → **GPU** (T4 is fine).

We train on cloud GPUs because local dev has no GPU; the code is device-agnostic,
so these are the same scripts that run locally on CPU. Download the resulting
`*.pth` checkpoints into local `artifacts/checkpoints/` to run the dashboard.

In [ ]:
!nvidia-smi -L

In [ ]:
!git clone https://github.com/Rayyan-mohammed/Urban-Route-Resilience.git
%cd Urban-Route-Resilience

In [ ]:
# Colab/Kaggle already ship a CUDA torch; install the rest + the package.
!pip install -q segmentation-models-pytorch timm albumentations omegaconf rich \
    rasterio geopandas osmnx scikit-image networkx pyproj scipy
!pip install -q -e .

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Build the dataset (OSMnx road masks, 4 terrains)
Reproduces the geo-referenced mask tiles + terrain-stratified split. Swap in real
SpaceNet/DeepGlobe/Cartosat imagery by setting `image_path` in the manifest later.

In [ ]:
!python scripts/build_dataset.py

## 2. Train the baseline (U-Net, Dice+Focal) — the control

In [ ]:
!python scripts/train.py -o train.num_workers=2 -o train.epochs=40

## 2b. Train D-LinkNet (Dice+Focal) — the strong CNN baseline
The DeepGlobe winner and the model most competitors reach for. Same trainer/loss
as U-Net (still pixel-loss, no topology objective) — this is the *strongest*
control our SegFormer+clDice model has to beat on the topology metrics.

In [ ]:
!python scripts/train.py \
    --config base.yaml data.yaml model_dlinknet.yaml train.yaml train_dlinknet.yaml \
    -o train.num_workers=2 -o train.epochs=40

## 3. Train SegFormer-B2 + clDice — the topology-aware model

In [ ]:
!python scripts/train.py \
    --config base.yaml data.yaml model_segformer.yaml train.yaml train_segformer.yaml \
    -o train.num_workers=2 -o train.epochs=40

## 4. Compare all three — the money table
Two comparisons:
- **U-Net vs SegFormer+clDice** — the classic control.
- **D-LinkNet vs SegFormer+clDice, both with TTA** — our USP against the *strongest* CNN baseline, with multi-scale + flip test-time augmentation applied to both models for a fair fight.

In both, watch IoU/Dice stay close while **clDice, connectivity-ratio and APLS** move in our favour.

In [ ]:
# U-Net vs SegFormer+clDice (classic control)
!python scripts/evaluate.py \
    --checkpoint artifacts/checkpoints/baseline_unet_best.pth \
    --compare    artifacts/checkpoints/segformer_cldice_best.pth \
    --apls

# Strongest baseline (D-LinkNet) vs SegFormer+clDice WITH multi-scale+flip TTA
!python scripts/evaluate.py \
    --checkpoint artifacts/checkpoints/dlinknet_best.pth \
    --compare    artifacts/checkpoints/segformer_cldice_best.pth \
    --apls --tta

## 5. Download the checkpoints
Place them in local `artifacts/checkpoints/` to run inference + the dashboard.

In [ ]:
try:
    from google.colab import files
    for ck in ('segformer_cldice_best.pth', 'dlinknet_best.pth', 'baseline_unet_best.pth'):
        files.download(f'artifacts/checkpoints/{ck}')
except Exception as e:
    print('Not on Colab or download unavailable:', e)